# Module 3: Quy hoạch toàn phương — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## QP đẳng thức: hệ KKT và null space (dạng A, B)

Hai cách phải cho cùng nghiệm.

In [1]:
import numpy as np
from scipy.linalg import null_space
Q = 2*np.eye(3); c = np.array([-2., -5, 0]); A = np.array([[1., 1, 1]]); b = np.array([3.])
K = np.block([[Q, A.T], [A, np.zeros((1, 1))]])
sol = np.linalg.solve(K, np.r_[-c, b]); print("KKT      x =", sol[:3].round(4), " mu =", sol[3].round(4))
Z = null_space(A); xp = np.array([3., 0, 0])
y = np.linalg.solve(Z.T @ Q @ Z, -Z.T @ (Q @ xp + c)); print("null space x =", (xp + Z @ y).round(4))

KKT      x = [ 0.8333  2.3333 -0.1667]  mu = 0.3333
null space x = [ 0.8333  2.3333 -0.1667]


## Tập hoạt động in bảng vết (dạng C)

Chạy active set cho ví dụ có x0 = (2,0), W0 = {3,5}.

In [2]:
import numpy as np
Q = 2*np.eye(2); c = np.array([-2., -5])
A = np.array([[-1, 2], [1, 2], [1, -2], [-1, 0], [0, -1.]]); b = np.array([2, 6, 2, 0, 0.])
x = np.array([2., 0]); W = [2, 4]
for k in range(10):
    g = Q @ x + c; Aw = A[W]
    K = np.block([[Q, Aw.T], [Aw, np.zeros((len(W), len(W)))]]) if W else Q
    s = np.linalg.solve(K, np.r_[-g, np.zeros(len(W))]); d, mu = s[:2], s[2:]
    print(k, "x =", x.round(3), "W =", [w + 1 for w in W], "d =", d.round(3), "mu =", mu.round(3))
    if np.linalg.norm(d) < 1e-9:
        if len(mu) == 0 or mu.min() >= -1e-9: break
        W.pop(int(np.argmin(mu)))
    else:
        al, blk = 1.0, None
        for i in range(len(b)):
            ad = A[i] @ d
            if i not in W and ad > 1e-12 and (b[i] - A[i] @ x) / ad < al: al, blk = (b[i] - A[i] @ x) / ad, i
        x = x + al * d
        if blk is not None: W.append(blk)
print("nghiệm", x, " f* =", round(float(np.sum((x - [1, 2.5]) ** 2)), 4))

0 x = [2. 0.] W = [3, 5] d = [0. 0.] mu = [-2. -1.]
1 x = [2. 0.] W = [5] d = [-1.  0.] mu = [-5.]
2 x = [1. 0.] W = [5] d = [-0.  0.] mu = [-5.]
3 x = [1. 0.] W = [] d = [-0.   2.5] mu = []
4 x = [1.  1.5] W = [1] d = [0.4 0.2] mu = [0.8]
5 x = [1.4 1.7] W = [1] d = [0. 0.] mu = [0.8]
nghiệm [1.4 1.7]  f* = 0.8


## Bài code · Hàm giải QP đẳng thức

Đầu vào: $Q,c,A,b$. Đầu ra: $(x,\mu,f)$ từ hệ KKT.

In [3]:
import numpy as np
def solve_eq_qp(Q, c, A, b):
    Q, c, A, b = (np.array(v, float) for v in (Q, c, A, b)); n, m = len(c), len(b)
    s = np.linalg.solve(np.block([[Q, A.T], [A, np.zeros((m, m))]]), np.r_[-c, b])
    return s[:n], s[n:], 0.5 * s[:n] @ Q @ s[:n] + c @ s[:n]
x, mu, f = solve_eq_qp(2*np.eye(3), [-2, -5, 0], [[1, 1, 1]], [3])
print("x =", x.round(4), " mu =", mu.round(4), " f =", round(float(f), 4))

x = [ 0.8333  2.3333 -0.1667]  mu = [0.3333]  f = -7.1667


## Bài code · Danh mục đầu tư (QP có ràng buộc)

Đầu vào: lợi suất kỳ vọng $\mu=(0.10,0.07,0.03)$, hiệp phương sai $\Sigma$ cho trước, lợi suất yêu cầu 6%. Đầu ra: tỷ trọng cực tiểu hoá phương sai với $\sum w=1,\ w\ge0$.

In [4]:
import numpy as np, cvxpy as cp
mu = np.array([0.10, 0.07, 0.03]); S = np.array([[0.09, 0.01, 0.0], [0.01, 0.04, 0.0], [0.0, 0.0, 0.01]])
w = cp.Variable(3)
p = cp.Problem(cp.Minimize(cp.quad_form(w, S)), [cp.sum(w) == 1, mu @ w >= 0.06, w >= 0]); p.solve()
print("w =", w.value.round(4), " phương sai =", round(p.value, 5), " lợi suất =", round(float(mu @ w.value), 4))

w = [0.2355 0.3379 0.4266]  phương sai = 0.01297  lợi suất = 0.06


## Bài code · QP không bị chặn dưới

Đầu ra: $\min -x_1^2+x_2$ s.t. $x\ge0$: kiểm hướng lùi xa và giá trị dọc $(t,0)$.

In [5]:
import numpy as np
Q = np.diag([-2., 0]); d = np.array([1., 0])
print("d'Qd =", d @ Q @ d, " f(t,0) với t=10,100,1000:", [-t**2 for t in (10, 100, 1000)])

d'Qd = -2.0  f(t,0) với t=10,100,1000: [-100, -10000, -1000000]


## Bài code · Đường đi active set bằng hàm

Đầu ra: dùng lại vòng lặp ở khối trên, trả về số vòng lặp và nghiệm; kiểm với cvxpy.

In [6]:
import numpy as np, cvxpy as cp
x = cp.Variable(2)
A = np.array([[-1, 2], [1, 2], [1, -2], [-1, 0], [0, -1.]]); b = np.array([2, 6, 2, 0, 0.])
p = cp.Problem(cp.Minimize(cp.sum_squares(x - np.array([1, 2.5]))), [A @ x <= b]); p.solve()
print("x* =", x.value.round(4), " f* =", round(p.value, 4), " lambda =", p.constraints[0].dual_value.round(3))

x* = [1.4 1.7]  f* = 0.8  lambda = [0.8 0.  0.  0.  0. ]
